# Part H: Model Explainability
This notebook explains the fitted pipeline produced by notebook 03; it does not retrain a model. The breast-cancer data set is used solely as a technical classification benchmark, not as a clinical or diagnostic system.

In [ ]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
import shap
from lime.lime_tabular import LimeTabularExplainer
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

np.random.seed(42)

## Pipeline and Test Data
The saved pipeline is loaded directly. The train/test split is recreated with the same stratification and seed, then pipeline preprocessing is transformed without fitting so SHAP and LIME use the classifier's trained feature space.

In [ ]:
pipeline = joblib.load('artifacts/model.joblib')
dataset = load_breast_cancer()
feature_names = list(dataset.feature_names)
class_names = list(dataset.target_names)
X_train, X_test, y_train, y_test = train_test_split(dataset.data, dataset.target, test_size=0.25, stratify=dataset.target, random_state=42)
preprocessor = pipeline[:-1]
classifier = pipeline.named_steps['classifier']
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)
predicted_labels = pipeline.predict(X_test)
predicted_probabilities = pipeline.predict_proba(X_test)[:, 1]

Positive SHAP values push the classifier score toward the positive class, while negative values push it away. Global explanations aggregate behavior across the test set; local explanations show why one individual prediction moved in a particular direction.

This output identifies one correctly classified record and a false-negative explanation record. The saved model has no default-threshold false negatives, so the fallback transparently selects the lowest-confidence true positive under a conservative threshold stress test.

In [ ]:
correct_index = int(np.flatnonzero(predicted_labels == y_test)[0])
false_negative_indices = np.flatnonzero((y_test == 1) & (predicted_labels == 0))
fallback_threshold = 0.70
if len(false_negative_indices):
    false_negative_index = int(false_negative_indices[0])
    false_negative_prediction = int(predicted_labels[false_negative_index])
    false_negative_note = 'Default-threshold false negative.'
else:
    positive_indices = np.flatnonzero(y_test == 1)
    false_negative_index = int(positive_indices[np.argmin(predicted_probabilities[positive_indices])])
    false_negative_prediction = int(predicted_probabilities[false_negative_index] >= fallback_threshold)
    false_negative_note = f'Conservative-threshold fallback at {fallback_threshold:.2f}; no default-threshold false negatives exist.'
print(f'Correct record: {correct_index}, true={int(y_test[correct_index])}, predicted={int(predicted_labels[correct_index])}')
print(f'FN explanation record: {false_negative_index}, true={int(y_test[false_negative_index])}, predicted={false_negative_prediction}')
print(false_negative_note)

## SHAP
TreeExplainer is applied to the extracted XGBoost classifier on data transformed by the pipeline's imputer and scaler. This preserves the exact feature representation used during fitting.

In [ ]:
shap_start = time.perf_counter()
tree_explainer = shap.TreeExplainer(classifier)
raw_shap_values = tree_explainer.shap_values(X_test_transformed)
shap_runtime_seconds = time.perf_counter() - shap_start
shap_values = np.asarray(raw_shap_values)
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]
base_value = float(np.asarray(tree_explainer.expected_value).reshape(-1)[-1])
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
importance_order = np.argsort(mean_abs_shap)[::-1]
top_5_indices = importance_order[:5]
top_5_text = [(feature_names[index], float(mean_abs_shap[index])) for index in top_5_indices]

This bar chart ranks every feature by mean absolute SHAP value, a global importance measure. Taller bars indicate features that changed model output more strongly on average across this test set.

In [ ]:
plt.figure(figsize=(10, 8))
plt.barh(np.array(feature_names)[importance_order][::-1], mean_abs_shap[importance_order][::-1])
plt.xlabel('Mean |SHAP value|')
plt.title('Global SHAP Feature Importance')
plt.show()

The bar chart gives a compact global ranking. These are learned associations within this fitted model, not causal effects: correlated features and confounding can change apparent importance without establishing a real-world mechanism.

This beeswarm summary plot shows both global importance and the direction and spread of SHAP effects for individual test records.

In [ ]:
shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names, show=True)

Each point is one record. Horizontal position shows whether a feature pushed toward or away from the positive class, while the distribution across rows shows how effects vary across records.

This waterfall plot explains the same correctly classified record selected above. It lists the largest feature pushes toward and away from the positive-class score.

In [ ]:
correct_explanation = shap.Explanation(values=shap_values[correct_index], base_values=base_value, data=X_test_transformed[correct_index], feature_names=feature_names)
shap.plots.waterfall(correct_explanation, max_display=10, show=True)

For this correctly classified record, positive bars push the score toward its positive class and negative bars push against it. The saved summary records the largest absolute feature contributions so the interpretation is available to downstream reviewers as structured data.

This waterfall plot explains the false-negative record when one exists at the default threshold; otherwise it explains the documented conservative-threshold fallback record. The focus is on negative SHAP contributions that reduced the positive-class score.

In [ ]:
false_negative_explanation = shap.Explanation(values=shap_values[false_negative_index], base_values=base_value, data=X_test_transformed[false_negative_index], feature_names=feature_names)
shap.plots.waterfall(false_negative_explanation, max_display=10, show=True)

The features with the strongest negative SHAP values are the signals that pushed this positive example toward a negative decision. In the fallback case, they identify which signals would create a false negative under the stricter operating threshold.

The next output prints the five globally most important features and their mean absolute SHAP values, which are also stored in the SHAP summary artifact.

In [ ]:
for feature, value in top_5_text:
    print(f'{feature}: {value:.6f}')

The printed values name the top five global features for this fitted pipeline. They quantify model reliance on this test-set explanation calculation, but do not establish that changing a feature would cause an outcome to change.

## LIME
LIME is fit on the transformed training data and uses the same correctly classified record as SHAP for a fair local comparison. The false-negative-style record is also reused so both methods address the same challenging case.

In [ ]:
lime_explainer = LimeTabularExplainer(X_train_transformed, feature_names=feature_names, class_names=class_names, mode='classification', random_state=42)
lime_start = time.perf_counter()
lime_correct = lime_explainer.explain_instance(X_test_transformed[correct_index], classifier.predict_proba, num_features=10, num_samples=1000)
lime_correct_runtime_seconds = time.perf_counter() - lime_start
lime_start = time.perf_counter()
lime_incorrect = lime_explainer.explain_instance(X_test_transformed[false_negative_index], classifier.predict_proba, num_features=10, num_samples=1000)
lime_incorrect_runtime_seconds = time.perf_counter() - lime_start
lime_correct_contributions = lime_correct.as_list(label=int(predicted_labels[correct_index]))
lime_incorrect_contributions = lime_incorrect.as_list(label=false_negative_prediction)

This LIME plot shows the top local surrogate contributions for the correctly classified record shared with SHAP.

In [ ]:
lime_correct.as_pyplot_figure(label=int(predicted_labels[correct_index]))
plt.show()

LIME approximates the classifier near this record with a simpler local model. Its displayed weights identify which transformed feature conditions supported or opposed the local predicted class.

This LIME plot explains the same false-negative or conservative-threshold fallback record used for the SHAP local explanation.

In [ ]:
lime_incorrect.as_pyplot_figure(label=false_negative_prediction)
plt.show()

The LIME weights show the local surrogate's conditions that supported the negative decision for this challenging positive record. Reusing the same index makes the SHAP–LIME comparison meaningful.

## SHAP and LIME Comparison
This output compares observed runtime and the most prominent shared-record features, rather than relying only on generic claims.

In [ ]:
shap_top_correct = [feature_names[index] for index in np.argsort(np.abs(shap_values[correct_index]))[::-1][:5]]
lime_top_correct = [feature for feature, _ in lime_correct_contributions[:5]]
print(f'SHAP runtime: {shap_runtime_seconds:.4f} seconds')
print(f'LIME correct-record runtime: {lime_correct_runtime_seconds:.4f} seconds')
print('SHAP top features:', shap_top_correct)
print('LIME top features:', lime_top_correct)

SHAP provides both the global ranking and local attributions through a unified game-theoretic framework, while LIME fits a local surrogate per record. TreeExplainer is exact for supported tree models, but that exactness does not extend to SHAP KernelExplainer or arbitrary models. LIME can change with kernel width or sample count, and its surrogate may not fully reflect the original model; SHAP can be expensive for non-tree models. The printed runtimes and shared-record feature lists show the observed cost and agreement or disagreement for this run.

The next cell writes clean JSON summaries for the downstream Explainability Reviewer Agent. All index and contribution values are cast to native Python types before serialization.

In [ ]:
def shap_contributions(record_index):
    order = np.argsort(np.abs(shap_values[record_index]))[::-1]
    return [{'feature': str(feature_names[index]), 'shap_value': float(shap_values[record_index, index])} for index in order]

shap_summary = {
    'global_importance': [
        {'feature': str(feature_names[index]), 'mean_abs_shap': float(mean_abs_shap[index])}
        for index in importance_order
    ],
    'top_5_features': [str(feature_names[index]) for index in top_5_indices],
    'local_correct_example': {
        'record_index': int(correct_index), 'true_label': int(y_test[correct_index]),
        'predicted_label': int(predicted_labels[correct_index]),
        'feature_contributions': shap_contributions(correct_index),
    },
    'local_false_negative_example': {
        'record_index': int(false_negative_index), 'true_label': int(y_test[false_negative_index]),
        'predicted_label': int(false_negative_prediction),
        'feature_contributions': shap_contributions(false_negative_index),
    },
}
lime_summary = {
    'correct_example': {
        'record_index': int(correct_index), 'true_label': int(y_test[correct_index]),
        'predicted_label': int(predicted_labels[correct_index]),
        'feature_contributions': [{'feature': str(feature), 'weight': float(weight)} for feature, weight in lime_correct_contributions],
    },
    'incorrect_example': {
        'record_index': int(false_negative_index), 'true_label': int(y_test[false_negative_index]),
        'predicted_label': int(false_negative_prediction),
        'feature_contributions': [{'feature': str(feature), 'weight': float(weight)} for feature, weight in lime_incorrect_contributions],
    },
}
artifact_directory = Path('artifacts')
artifact_directory.mkdir(exist_ok=True)
with (artifact_directory / 'shap_summary.json').open('w', encoding='utf-8') as shap_file:
    json.dump(shap_summary, shap_file, indent=2)
with (artifact_directory / 'lime_summary.json').open('w', encoding='utf-8') as lime_file:
    json.dump(lime_summary, lime_file, indent=2)

The SHAP artifact contains all global feature importances plus structured local contributions for the correct and false-negative-style records. The LIME artifact stores the matching local surrogate contributions, allowing the downstream agent to review explanations without parsing plots.